In [ ]:
df = 0 

In [ ]:
def iqr_filter(group):

    q1 = group['month_spend'].quantile(0.25)
    q3 = group['month_spend'].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[
        group['month_spend'].between(lower, upper)
    ]


clean_df = (
    df
    .groupby('campaigns_cnt', group_keys=False)
    .apply(iqr_filter)
    .reset_index(drop=True)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# -----------------------------
# Форматирование оси Y
# -----------------------------

def format_y_axis():

    plt.gca().yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f'{x:,.0f}'.replace(',', ' ')
        )
    )


# =========================================================
# 1. РТО когорт по календарным месяцам
# =========================================================

calendar_stats = (
    clean_df
    .groupby(
        ['campaigns_cnt', 'month_dt'],
        as_index=False
    )
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median')
    )
)

calendar_stats = calendar_stats.sort_values(
    ['campaigns_cnt', 'month_dt']
)


# -----------------------------
# Медианный РТО - календарные месяцы
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(calendar_stats['campaigns_cnt'].unique()):

    part = calendar_stats[
        calendar_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['month_dt'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.title('Медианный РТО когорт по календарным месяцам')

plt.xlabel('Месяц')
plt.ylabel('Медианный РТО')

plt.grid(True, alpha=0.3)

plt.legend(
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

format_y_axis()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================================================
# 2. Поведение относительно первой кампании
# =========================================================

shift_stats = (
    clean_df
    .groupby(
        ['campaigns_cnt', 'month_shift'],
        as_index=False
    )
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median')
    )
)

shift_stats = shift_stats.sort_values(
    ['campaigns_cnt', 'month_shift']
)


# -----------------------------
# Медианный РТО - относительно первой кампании
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(shift_stats['campaigns_cnt'].unique()):

    part = shift_stats[
        shift_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['month_shift'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я камп.',
        '+1',
        '+2',
        '+3',
        '+4'
    ]
)

plt.title(
    'Медианный РТО относительно первой кампании'
)

plt.xlabel('Период относительно первой кампании')
plt.ylabel('Медианный РТО')

plt.grid(True, alpha=0.3)

plt.legend(
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

format_y_axis()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


shift_stats = (
    clean_df
    .groupby(['campaigns_cnt', 'month_shift'], as_index=False)
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median')
    )
    .sort_values(['campaigns_cnt', 'month_shift'])
)


def plot_rto_by_metric(ax, data, metric, title):
    for cohort in sorted(data['campaigns_cnt'].unique()):
        part = data[data['campaigns_cnt'] == cohort]

        ax.plot(
            part['month_shift'],
            part[metric],
            marker='o',
            linewidth=2,
            label=f'{cohort} камп.'
        )

    ax.axvline(x=0, linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('Период относительно первой кампании')
    ax.grid(True, alpha=0.3)

    ax.set_xticks([-1, 0, 1, 2, 3, 4])
    ax.set_xticklabels(['-1 мес', '1-я камп.', '+1', '+2', '+3', '+4'])

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', ' '))
    )


fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

plot_rto_by_metric(
    axes[0],
    shift_stats,
    'median_spend_per_client',
    'Медианный РТО'
)

plot_rto_by_metric(
    axes[1],
    shift_stats,
    'avg_spend_per_client',
    'Средний РТО'
)

axes[0].set_ylabel('РТО')

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 0.95),
    loc='upper left'
)

fig.suptitle('РТО по когортам относительно первой кампании', fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
select
    c.client_id,
    c.campaigns_cnt,
    c.first_campaign_dt,

    date_trunc('month', ch.datetime)::date as month_dt,

    (
        extract(year from age(
            date_trunc('month', ch.datetime),
            c.first_campaign_month
        )) * 12

        +

        extract(month from age(
            date_trunc('month', ch.datetime),
            c.first_campaign_month
        ))

    )::int as month_shift,

    sum(ch.summ_discounted) as month_spend

from cohorts c

join cheque_filtered ch
    on ch.contact_id = c.client_id
    and ch.datetime >= c.first_campaign_month - interval '1 month'
    and ch.datetime < date '{date_end}' + interval '1 day'

group by
    c.client_id,
    c.campaigns_cnt,
    c.first_campaign_dt,
    month_dt,
    month_shift

order by
    c.client_id,
    month_shift;